# ViMamba-SER | Phase 2 — Audio-Only Baseline (WavLM + MLP)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/ViMamba-SER-outputs"
os.environ["DRIVE_DIR"] = DRIVE_DIR

if not os.path.exists("/content/ViMamba-SER"):
    !git clone https://github.com/QuangHuy1911/ViMamba-SER.git /content/ViMamba-SER
else:
    !git -C /content/ViMamba-SER pull

%cd /content/ViMamba-SER
!pip install -r requirements.txt -q
print("\u2705 Setup done")

In [ ]:
import sys
sys.path.append("/content/ViMamba-SER/src")
from config import *

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from transformers import WavLMModel, Wav2Vec2FeatureExtractor
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"\u2705 Device: {DEVICE}")


## Step 1 — Load splits.json từ Phase 1

In [ ]:
splits_path = f"{EMBED_DIR}/splits.json"
assert os.path.exists(splits_path), "\u274c Ch\u1ea1y Phase 1 tr\u01b0\u1edbc!"

with open(splits_path) as f:
    splits = json.load(f)

train_idx = splits["train"]
val_idx   = splits["val"]
test_idx  = splits["test"]

print(f"\u2705 Splits loaded: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

## Step 2 — Load dataset + cast audio

In [ ]:
print("Loading ViSEC...")
ds = load_dataset(DATASET_NAME, split="train")
ds = ds.cast_column("path", Audio(sampling_rate=SAMPLE_RATE))
print(f"\u2705 Dataset: {len(ds)} samples")

## Step 3 — Extract WavLM embeddings (frozen, mean-pooled)

In [ ]:
print(f"Loading WavLM from {WAVLM_CKPT}...")
processor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_CKPT)
wavlm     = WavLMModel.from_pretrained(WAVLM_CKPT).to(DEVICE)
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False
print("\u2705 WavLM loaded and frozen")


In [ ]:
embed_path = f"{EMBED_DIR}/wavlm_embeddings.npy"
label_path = f"{EMBED_DIR}/wavlm_labels.npy"

if os.path.exists(embed_path) and os.path.exists(label_path):
    print("\u2705 Embeddings already exist, loading from Drive...")
    embeddings = np.load(embed_path)
    labels     = np.load(label_path)
    # Verify loaded labels are not corrupted
    from collections import Counter
    dist = Counter(labels.tolist())
    print(f"   Label distribution: {dist}")
    assert -1 not in labels and min(dist.keys()) >= 0, "\u274c Labels corrupted! Delete files and re-extract."
else:
    all_indices = list(range(len(ds)))
    embeddings  = np.zeros((len(ds), AUDIO_DIM), dtype=np.float32)
    labels      = np.zeros(len(ds), dtype=np.int64) - 1  # -1 to detect unprocessed

    with torch.no_grad():
        for i in tqdm(all_indices, desc="Extracting WavLM"):
            item   = ds[i]
            array  = item["path"]["array"]
            inputs = processor(
                array,
                sampling_rate=SAMPLE_RATE,
                return_tensors="pt",
                padding=True
            ).to(DEVICE)
            out = wavlm(**inputs)
            emb = out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings[i] = emb
            labels[i]     = LABEL_MAP[item["emotion"]]

            if (i + 1) % 500 == 0:
                np.save(embed_path, embeddings)
                np.save(label_path, labels)
                print(f"  \U0001f4be Checkpoint at {i+1}/{len(ds)}")

    np.save(embed_path, embeddings)
    np.save(label_path, labels)

    from collections import Counter
    dist = Counter(labels.tolist())
    print(f"\n\u2705 Done. Label distribution: {dist}")
    assert -1 not in labels, "\u274c C\u00f3 sample ch\u01b0a x\u1eed l\u00fd!"

assert embeddings.shape == (len(ds), AUDIO_DIM)
assert labels.shape == (len(ds),)
print(f"\u2705 embeddings: {embeddings.shape}, labels OK")


## Step 4 — Dataset + DataLoader

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels, indices):
        self.X = torch.tensor(embeddings[indices], dtype=torch.float32)
        self.y = torch.tensor([labels[i] for i in indices], dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = EmbeddingDataset(embeddings, labels, train_idx)
val_ds   = EmbeddingDataset(embeddings, labels, val_idx)
test_ds  = EmbeddingDataset(embeddings, labels, test_idx)

train_loader = DataLoader(train_ds, batch_size=P2_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=P2_BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=P2_BATCH_SIZE)

print(f"\u2705 DataLoaders ready")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches  : {len(val_loader)}")

## Step 5 — MLP Classifier

In [ ]:
class AudioMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(AUDIO_DIM, P2_HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(P2_DROPOUT),
            nn.Linear(P2_HIDDEN_DIM, P2_HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(P2_DROPOUT),
            nn.Linear(P2_HIDDEN_DIM // 2, NUM_CLASSES)
        )
    def forward(self, x):
        return self.net(x)

model     = AudioMLP().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=P2_LR)
criterion = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in model.parameters())
print(f"\u2705 Model ready | params: {total_params:,}")

## Step 6 — Training

In [ ]:
TRAIN_EPOCHS = 50  # override P2_EPOCHS for better convergence
best_val_acc = 0.0
train_losses, val_accs = [], []

for epoch in range(1, TRAIN_EPOCHS + 1):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            preds = model(X_batch.to(DEVICE)).argmax(dim=1).cpu()
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    val_acc = correct / total

    train_losses.append(total_loss / len(train_loader))
    val_accs.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f"{RUNS_DIR}/phase2_best.pt")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{TRAIN_EPOCHS} | loss={train_losses[-1]:.4f} | val_acc={val_acc:.4f} | best={best_val_acc:.4f}")

print(f"\n\u2705 Training done | Best val acc: {best_val_acc:.4f}")


## Step 7 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title("Train Loss"); ax1.set_xlabel("Epoch")
ax2.plot(val_accs);     ax2.set_title("Val Accuracy"); ax2.set_xlabel("Epoch")
plt.tight_layout()
save_path = f"{FIGURES_DIR}/phase2_training_curves.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\u2705 Saved: {save_path}")

## Step 8 — Test Evaluation

In [ ]:
model.load_state_dict(torch.load(f"{RUNS_DIR}/phase2_best.pt", map_location=DEVICE))
model.eval()

all_preds, all_labels_test = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels_test.extend(y_batch.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels_test)) / len(all_labels_test)
print(f"\U0001f3af Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print()
print(classification_report(all_labels_test, all_preds, target_names=LABEL_NAMES))

# Confusion matrix
cm = confusion_matrix(all_labels_test, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES,
            yticklabels=LABEL_NAMES, cmap="Blues", ax=ax)
ax.set_title(f"Phase 2 \u2014 Audio-Only Confusion Matrix\nTest Acc: {test_acc*100:.2f}%")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
save_path = f"{FIGURES_DIR}/phase2_confusion_matrix.png"
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\u2705 Saved: {save_path}")

## \u2705 Phase 2 Checklist

In [ ]:
checks = {
    "Embeddings extracted": os.path.exists(f"{EMBED_DIR}/wavlm_embeddings.npy"),
    "Best model saved"    : os.path.exists(f"{RUNS_DIR}/phase2_best.pt"),
    "Training curves saved": os.path.exists(f"{FIGURES_DIR}/phase2_training_curves.png"),
    "Confusion matrix saved": os.path.exists(f"{FIGURES_DIR}/phase2_confusion_matrix.png"),
    "Test acc > 50%"      : test_acc > 0.50,
}

print("=" * 45)
print("       Phase 2 \u2014 Final Checklist")
print("=" * 45)
all_passed = True
for item, passed in checks.items():
    icon = "\u2705" if passed else "\u274c"
    print(f"  {icon} {item}")
    if not passed:
        all_passed = False

print("=" * 45)
print(f"\n\U0001f3af Phase 2 Audio-Only Test Accuracy: {test_acc*100:.2f}%")
if all_passed:
    print("\U0001f389 Phase 2 PASSED \u2014 G\u1edfi screenshot n\u00e0y cho Quang Huy")
    print("   Ghi nh\u1edb con s\u1ed1 Test Accuracy n\u00e0y \u0111\u1ec3 so s\u00e1nh v\u1edbi Phase 3!")
else:
    print("\u26a0\ufe0f  C\u00f3 item ch\u01b0a pass \u2014 g\u1edfi screenshot cho Quang Huy")